# 1. Dataset Overview

In [97]:
import os
import numpy as np
import pandas as pd
#print(os.path.exists("D:\college ml project\individual+household+electric+power+consumption"))
# sep=';' used for sepration in the dataset if not then it is not properly aligned 
df=pd.read_csv("D:\college ml project\individual+household+electric+power+consumption\household_power_consumption.txt",sep=';')
df.head()

C:\Users\adity\AppData\Local\Temp\ipykernel_20268\3278896439.py:6: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("D:\college ml project\individual+household+electric+power+consumption\household_power_consumption.txt",sep=';')


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


# 2.No of rows and no of columns to get the overview

In [98]:
df.shape

(2075259, 9)

# The dataset contains 2,075,259 observations and 9 columns.

Each row represents an electricity consumption observation recorded at one-minute intervals.

Date and Time provide temporal information.

The remaining seven columns contain household electricity measurements.

Initial observation: The dataset is large and contains both temporal and numerical information, making it suitable for time-series analysis and forecasting.

# 3.Understanding Data Types

In [99]:
df.dtypes

Date                      object
Time                      object
Global_active_power       object
Global_reactive_power     object
Voltage                   object
Global_intensity          object
Sub_metering_1            object
Sub_metering_2            object
Sub_metering_3           float64
dtype: object

# The initial data type inspection shows:

8 columns have the object data type.

1 column (Sub_metering_3) has the float64 data type.

Date and Time are stored as objects, which is expected before datetime conversion.

The measurement columns are mostly stored as objects because they contain the missing-value marker ?.

Observation: The numerical measurement columns need appropriate type conversion before machine learning.

# 3. Initial Missing-Value Inspection

In [100]:
df.isnull().sum()


Date                         0
Time                         0
Global_active_power          0
Global_reactive_power        0
Voltage                      0
Global_intensity             0
Sub_metering_1               0
Sub_metering_2               0
Sub_metering_3           25979
dtype: int64

# 4.Using isnull().sum(), we identified 25,979 missing values in Sub_metering_3.

However, this does not represent all missing observations in the dataset.

Pandas detects values such as NaN and None as missing values, but does not automatically recognize the string ? as a missing value.

Therefore, a separate inspection is required to identify these markers.

In [101]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   object 
 1   Time                   object 
 2   Global_active_power    object 
 3   Global_reactive_power  object 
 4   Voltage                object 
 5   Global_intensity       object 
 6   Sub_metering_1         object 
 7   Sub_metering_2         object 
 8   Sub_metering_3         float64
dtypes: float64(1), object(8)
memory usage: 142.5+ MB


# 4. Identifying Missing-Value Markers

In [102]:
df.eq("?").sum()

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3               0
dtype: int64

# 5.The eq("?").sum() operation identifies the number of question-mark markers in each column.

Observations:

Six measurement columns contain 25,979 question-mark markers each.

Date and Time contain no question-mark markers.

Sub_metering_3 contains no question-mark markers, but has 25,979 actual NaN values.

Conclusion: The dataset contains missing values represented in two different ways: ? and NaN. These need to be standardized during preprocessing.

# 6.Missing Count Percentage


In [103]:
misscountNaN=(df.isnull().sum()/df.shape[0])*100
misscountother=(df.eq("?").sum()/df.shape[0])*100
print(misscountNaN)
print(misscountother)

Date                     0.000000
Time                     0.000000
Global_active_power      0.000000
Global_reactive_power    0.000000
Voltage                  0.000000
Global_intensity         0.000000
Sub_metering_1           0.000000
Sub_metering_2           0.000000
Sub_metering_3           1.251844
dtype: float64
Date                     0.000000
Time                     0.000000
Global_active_power      1.251844
Global_reactive_power    1.251844
Voltage                  1.251844
Global_intensity         1.251844
Sub_metering_1           1.251844
Sub_metering_2           1.251844
Sub_metering_3           0.000000
dtype: float64


# The percentage of missing values is calculated separately for:

Actual missing values (NaN).

Question-mark markers (?).

The results show approximately 1.25% missing observations in the affected measurement columns for each representation.

This analysis helps quantify the extent of missing data before selecting an appropriate treatment strategy.

# Inspecting missing observations

In [104]:
df.loc[:,'Global_active_power'].eq("?").sum()

np.int64(25979)

In [105]:
df.loc[df['Global_active_power']=='?',df.columns].head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
6839,21/12/2006,11:23:00,?,?,?,?,?,?,NaN
6840,21/12/2006,11:24:00,?,?,?,?,?,?,NaN
19724,30/12/2006,10:08:00,?,?,?,?,?,?,NaN
19725,30/12/2006,10:09:00,?,?,?,?,?,?,NaN
41832,14/1/2007,18:36:00,?,?,?,?,?,?,NaN


In [106]:
df.loc[df['Global_active_power']=='?']

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
6839,21/12/2006,11:23:00,?,?,?,?,?,?,NaN
6840,21/12/2006,11:24:00,?,?,?,?,?,?,NaN
19724,30/12/2006,10:08:00,?,?,?,?,?,?,NaN
19725,30/12/2006,10:09:00,?,?,?,?,?,?,NaN
41832,14/1/2007,18:36:00,?,?,?,?,?,?,NaN
...,...,...,...,...,...,...,...,...,...
1990185,28/9/2010,19:09:00,?,?,?,?,?,?,NaN
1990186,28/9/2010,19:10:00,?,?,?,?,?,?,NaN
1990187,28/9/2010,19:11:00,?,?,?,?,?,?,NaN
1990188,28/9/2010,19:12:00,?,?,?,?,?,?,NaN


In [107]:
df.loc[6835:6843]

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
6835,21/12/2006,11:19:00,0.244,0.000,242.050,1.000,0.000,0.000,0.0
6836,21/12/2006,11:20:00,0.244,0.000,242.080,1.000,0.000,0.000,0.0
6837,21/12/2006,11:21:00,0.242,0.000,241.670,1.000,0.000,0.000,0.0
6838,21/12/2006,11:22:00,0.244,0.000,242.290,1.000,0.000,0.000,0.0
6839,21/12/2006,11:23:00,?,?,?,?,?,?,NaN
6840,21/12/2006,11:24:00,?,?,?,?,?,?,NaN
6841,21/12/2006,11:25:00,0.246,0.000,241.740,1.000,0.000,0.000,0.0
6842,21/12/2006,11:26:00,0.246,0.000,241.830,1.000,0.000,0.000,0.0
6843,21/12/2006,11:27:00,0.244,0.000,240.960,1.000,0.000,0.000,0.0


# 7.Inspecting Missing Observations
The rows containing missing markers were inspected using Pandas indexing.

The observations show that multiple measurement columns can be missing at the same timestamp.

Some missing observations also occur consecutively.

Initial finding: Missing values may occur in continuous time intervals rather than only as isolated observations.

Further investigation of missing-gap lengths is required before selecting an imputation strategy.

# 8.changing "?" to NaN

In [111]:
measurement_cols=df.columns[2:]
df[measurement_cols]=df[measurement_cols].replace('?',np.nan)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0
...,...,...,...,...,...,...,...,...,...
2075254,26/11/2010,20:58:00,0.946,0.0,240.43,4.0,0.0,0.0,0.0
2075255,26/11/2010,20:59:00,0.944,0.0,240.0,4.0,0.0,0.0,0.0
2075256,26/11/2010,21:00:00,0.938,0.0,239.82,3.8,0.0,0.0,0.0
2075257,26/11/2010,21:01:00,0.934,0.0,239.7,3.8,0.0,0.0,0.0


# Convert measurement columns into numeric format

In [114]:
for col in measurement_cols:
    df[col]=pd.to_numeric(df[col],errors='coerce')


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


# Identify rows containing at least one missing measurement

In [172]:
missing_rows=df[measurement_cols].isna().any(axis=1)
missing_rows

0          False
1          False
2          False
3          False
4          False
           ...  
2075254    False
2075255    False
2075256    False
2075257    False
2075258    False
Length: 2075259, dtype: bool

In [173]:
# Assign IDs to consecutive missing-value blocks
group_id = missing_rows.ne(missing_rows.shift()).cumsum()

# Summarize missing-value blocks
gap_summary = (
    df.loc[missing_rows]
    .assign(group_id=group_id[missing_rows])
    .groupby('group_id')
    .agg(
        start_date=('Date', 'first'),
        start_time=('Time', 'first'),
        end_date=('Date', 'last'),
        end_time=('Time', 'last'),
        missing_minutes=('Date', 'size')
    )
    .reset_index(drop=True)
)

display(gap_summary.head(10))

print("Total missing blocks:", len(gap_summary))
print("Longest missing block:", gap_summary['missing_minutes'].max(), "minutes")

,start_date,start_time,end_date,end_time,missing_minutes
0,21/12/2006,11:23:00,21/12/2006,11:24:00,2
1,30/12/2006,10:08:00,30/12/2006,10:09:00,2
2,14/1/2007,18:36:00,14/1/2007,18:36:00,1
3,28/1/2007,17:13:00,28/1/2007,17:13:00,1
4,22/2/2007,22:58:00,22/2/2007,22:59:00,2
5,25/3/2007,17:52:00,25/3/2007,17:52:00,1
6,28/4/2007,00:21:00,30/4/2007,14:23:00,3723
7,1/6/2007,19:14:00,1/6/2007,19:14:00,1
8,6/6/2007,21:56:00,6/6/2007,21:56:00,1
9,9/6/2007,11:44:00,9/6/2007,11:44:00,1


Total missing blocks: 71
Longest missing block: 7226 minutes


In [174]:
gap_summary.sort_values(
    by='missing_minutes',
    ascending=False
).head(10)



,start_date,start_time,end_date,end_time,missing_minutes
68,17/8/2010,21:02:00,22/8/2010,21:27:00,7226
69,25/9/2010,03:56:00,28/9/2010,19:12:00,5237
6,28/4/2007,00:21:00,30/4/2007,14:23:00,3723
49,13/6/2009,00:30:00,15/6/2009,07:34:00,3305
58,12/1/2010,14:53:00,14/1/2010,19:01:00,3129
62,20/3/2010,03:52:00,21/3/2010,13:38:00,2027
51,13/8/2009,05:00:00,13/8/2009,19:50:00,891
15,15/7/2007,16:49:00,15/7/2007,18:11:00,83
38,10/12/2008,10:48:00,10/12/2008,11:57:00,70
16,15/7/2007,18:21:00,15/7/2007,19:07:00,47


In [175]:
print("Missing values per column:")
display(df[measurement_cols].isna().sum())

Missing values per column:


Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

In [176]:
missing_pattern = df[measurement_cols].isna()

print("Rows where all 7 columns are missing:",
      missing_pattern.all(axis=1).sum())

print("Rows where only some columns are missing:",
      (missing_pattern.any(axis=1) &
       ~missing_pattern.all(axis=1)).sum())

print("Total rows containing missing values:",
      missing_pattern.any(axis=1).sum())

Rows where all 7 columns are missing: 25979
Rows where only some columns are missing: 0
Total rows containing missing values: 25979


# Date + Time → Timestamp

In [177]:
df['timestamp']=df['Date']+' '+df['Time']
df['timestamp']=pd.to_datetime(df['timestamp'],format='%d/%m/%Y %H:%M:%S')
df.dtypes
df['timestamp'].head()

0   2006-12-16 17:24:00
1   2006-12-16 17:25:00
2   2006-12-16 17:26:00
3   2006-12-16 17:27:00
4   2006-12-16 17:28:00
Name: timestamp, dtype: datetime64[ns]

# Why Do We Create a Timestamp Column and Sort the Dataset?

The original dataset contains two separate columns: Date and Time. We combine these columns into a single timestamp column to represent the exact date and time of each electricity consumption observation.

Why create a timestamp column?

Easy time-based operations: A single datetime column makes it easier to perform time-based calculations and comparisons.

Temporal feature extraction: We can extract useful features such as hour, day, month, weekday and weekend.

Time-series analysis: It helps us identify consumption patterns across different time periods.

Lag feature creation: It provides a reliable time reference when creating features from previous observations.

# Sorting the data based on the timestamp

In [178]:
df.sort_values(by='timestamp',ascending=True,inplace=True)

# Why sort the dataset by timestamp?

The dataset contains time-series observations, so the chronological order of records is important.

Sorting the dataset from the earliest timestamp to the latest timestamp ensures that previous rows represent earlier observations.

In [179]:
df

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,timestamp,missing_rows
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00,0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00,0
...,...,...,...,...,...,...,...,...,...,...,...
2075254,26/11/2010,20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0,2010-11-26 20:58:00,0
2075255,26/11/2010,20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0,2010-11-26 20:59:00,0
2075256,26/11/2010,21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0,2010-11-26 21:00:00,0
2075257,26/11/2010,21:01:00,0.934,0.000,239.70,3.8,0.0,0.0,0.0,2010-11-26 21:01:00,0


In [180]:
df['timestamp'].duplicated().sum()

np.int64(0)

In [181]:
df['timestamp'].diff()

0                     NaT
1         0 days 00:01:00
2         0 days 00:01:00
3         0 days 00:01:00
4         0 days 00:01:00
                ...      
2075254   0 days 00:01:00
2075255   0 days 00:01:00
2075256   0 days 00:01:00
2075257   0 days 00:01:00
2075258   0 days 00:01:00
Name: timestamp, Length: 2075259, dtype: timedelta64[ns]

In [182]:
timestamp_diff=df['timestamp'].diff()
count=0
for col in timestamp_diff:
    if(col>pd.Timedelta(minutes=1)):
        count+=1
print(count)

0


### Checking Time-Series Frequency and Missing Timestamps

The dataset contains electricity consumption observations recorded at one-minute intervals. To verify the continuity of the time series, we calculated the difference between consecutive timestamps using the `diff()` method.

**Observations:**

- Total observations: 2,075,259
- Consecutive one-minute intervals: 2,075,258
- Maximum time difference: 1 minute
- Intervals greater than one minute: 0
- Duplicate timestamps: 0

**Conclusion:**

The dataset has continuous one-minute timestamp intervals, with no missing timestamps or duplicate timestamps. Therefore, the temporal sequence is suitable for creating time-based lag features after handling missing measurement values.

In [209]:

missing_rows=missing_rows.replace({True: 1, False: 0})
missing_rows

0          0
1          0
2          0
3          0
4          0
          ..
2075254    0
2075255    0
2075256    0
2075257    0
2075258    0
Length: 2075259, dtype: int64

In [211]:
df.drop(columns='missing_rows', inplace=True)
df

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,timestamp
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00
...,...,...,...,...,...,...,...,...,...,...
2075254,26/11/2010,20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0,2010-11-26 20:58:00
2075255,26/11/2010,20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0,2010-11-26 20:59:00
2075256,26/11/2010,21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0,2010-11-26 21:00:00
2075257,26/11/2010,21:01:00,0.934,0.000,239.70,3.8,0.0,0.0,0.0,2010-11-26 21:01:00


In [212]:
missing_diff = missing_rows.diff()
missing_diff

0          NaN
1          0.0
2          0.0
3          0.0
4          0.0
          ... 
2075254    0.0
2075255    0.0
2075256    0.0
2075257    0.0
2075258    0.0
Length: 2075259, dtype: float64

In [213]:
start = None
gap_lengths = []
for idx, value in enumerate(missing_diff):
    if value == 1:
        start = idx

    elif value == -1:
        if start is not None:
            gap_length = idx - start
            gap_lengths.append(gap_length)
            start = None
short_gaps = [x for x in gap_lengths if 1 <= x <= 5]
medium_gaps = [x for x in gap_lengths if 6 <= x <= 60]
long_gaps = [x for x in gap_lengths if x > 60]
print(gap_lengths)
print(len(short_gaps))
print(len(medium_gaps))
print(len(long_gaps))

[2, 2, 1, 1, 2, 1, 3723, 1, 1, 1, 1, 3, 33, 2, 1, 83, 47, 1, 21, 1, 2, 2, 1, 1, 1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 43, 6, 2, 1, 70, 1, 1, 38, 2, 24, 1, 1, 2, 1, 3, 3305, 4, 891, 1, 2, 1, 1, 2, 1, 3129, 1, 1, 1, 2027, 1, 1, 1, 1, 1, 7226, 5237, 1]
55
7
9


In [214]:
short_df=pd.DataFrame(short_gaps)
print(short_df.max())
print(short_df.min())
print(short_df.mean())

0    4
dtype: int64
0    1
dtype: int64
0    1.381818
dtype: float64


In [215]:
df_copy=df.copy()

In [216]:
print("Original shape:", df.shape)
print("Copy shape:", df_copy.shape)

print("Original missing values:", df.isna().sum().sum())
print("Copy missing values:", df_copy.isna().sum().sum())

Original shape: (2075259, 10)
Copy shape: (2075259, 10)
Original missing values: 181853
Copy missing values: 181853


SyntaxError: incomplete input (2420292780.py, line 1)